In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow

In [2]:
df= pd.read_csv('cust_satisfaction.csv')
df.head()

,Gender,Customer Type,Type of Travel,Class,satisfaction,Age,Flight Distance,Inflight entertainment,Baggage handling,Cleanliness,Departure Delay in Minutes,Arrival Delay in Minutes
0,Male,Loyal Customer,Personal Travel,Eco Plus,neutral or dissatisfied,13,460,5,4,5,25,18.0
1,Male,disloyal Customer,Business travel,Business,neutral or dissatisfied,25,235,1,3,1,1,6.0
2,Female,Loyal Customer,Business travel,Business,satisfied,26,1142,5,4,5,0,0.0
3,Female,Loyal Customer,Business travel,Business,neutral or dissatisfied,25,562,2,3,2,11,9.0
4,Male,Loyal Customer,Business travel,Business,satisfied,61,214,3,4,3,0,0.0


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103904 entries, 0 to 103903
Data columns (total 12 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   Gender                      103904 non-null  object 
 1   Customer Type               103904 non-null  object 
 2   Type of Travel              103904 non-null  object 
 3   Class                       103904 non-null  object 
 4   satisfaction                103904 non-null  object 
 5   Age                         103904 non-null  int64  
 6   Flight Distance             103904 non-null  int64  
 7   Inflight entertainment      103904 non-null  int64  
 8   Baggage handling            103904 non-null  int64  
 9   Cleanliness                 103904 non-null  int64  
 10  Departure Delay in Minutes  103904 non-null  int64  
 11  Arrival Delay in Minutes    103594 non-null  float64
dtypes: float64(1), int64(6), object(5)
memory usage: 9.5+ MB


In [4]:
df.isnull().sum()
df.dropna(inplace=True)


In [5]:

df.duplicated().sum()
df.drop_duplicates(inplace=True)

In [6]:
df["Customer Type"].value_counts()

Customer Type
Loyal Customer       84517
disloyal Customer    18905
Name: count, dtype: int64

In [7]:
loyal_customer = df[df["Customer Type"] == "Loyal Customer"]
disloyal_customer = df[df["Customer Type"] == "disloyal Customer"]

In [8]:
loyal_customer=loyal_customer.sample(20000)
loyal_customer.shape

(20000, 12)

In [9]:
balanced_df = pd.concat([loyal_customer, disloyal_customer], axis=0)
balanced_df.shape

(38905, 12)

In [10]:
cat_col= balanced_df.select_dtypes(include=['object'])
cat_col.head()

,Gender,Customer Type,Type of Travel,Class,satisfaction
1832,Female,Loyal Customer,Personal Travel,Business,neutral or dissatisfied
50907,Female,Loyal Customer,Business travel,Business,satisfied
45114,Female,Loyal Customer,Personal Travel,Eco Plus,neutral or dissatisfied
11601,Male,Loyal Customer,Business travel,Business,neutral or dissatisfied
30925,Female,Loyal Customer,Personal Travel,Eco,satisfied


In [11]:
num_col= balanced_df.select_dtypes(exclude=['object'])
num_col.head()

,Age,Flight Distance,Inflight entertainment,Baggage handling,Cleanliness,Departure Delay in Minutes,Arrival Delay in Minutes
1832,15,417,1,3,1,8,7.0
50907,23,3832,4,4,4,0,2.0
45114,36,692,4,5,3,163,142.0
11601,45,2163,3,3,2,5,23.0
30925,25,2607,2,3,2,0,0.0


In [12]:
pd.get_dummies(cat_col, drop_first=True).astype(int).head()

,Gender_Male,Customer Type_disloyal Customer,Type of Travel_Personal Travel,Class_Eco,Class_Eco Plus,satisfaction_satisfied
1832,0,0,1,0,0,0
50907,0,0,0,0,0,1
45114,0,0,1,0,1,0
11601,1,0,0,0,0,0
30925,0,0,1,1,0,1


In [13]:
# ## one hot encoding
# cat_col = pd.get_dummies(cat_col, drop_first=True).astype(int)           # drop_first=True avoids dummy variable trap
# cat_col

In [14]:
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
ohe= OneHotEncoder(drop="if_binary")
cat_col_encoded = ohe.fit_transform(cat_col).toarray()
cat_col_encoded

array([[0., 0., 1., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 1.],
       [0., 0., 1., ..., 0., 1., 0.],
       ...,
       [0., 1., 0., ..., 1., 0., 0.],
       [1., 1., 0., ..., 0., 0., 0.],
       [0., 1., 0., ..., 1., 0., 0.]])

In [15]:
column_name=list(ohe.get_feature_names_out())
column_name

['Gender_Male',
 'Customer Type_disloyal Customer',
 'Type of Travel_Personal Travel',
 'Class_Business',
 'Class_Eco',
 'Class_Eco Plus',
 'satisfaction_satisfied']

In [16]:
one_hot = pd.DataFrame(cat_col_encoded,columns=column_name)
one_hot.head()

,Gender_Male,Customer Type_disloyal Customer,Type of Travel_Personal Travel,Class_Business,Class_Eco,Class_Eco Plus,satisfaction_satisfied
0,0.0,0.0,1.0,1.0,0.0,0.0,0.0
1,0.0,0.0,0.0,1.0,0.0,0.0,1.0
2,0.0,0.0,1.0,0.0,0.0,1.0,0.0
3,1.0,0.0,0.0,1.0,0.0,0.0,0.0
4,0.0,0.0,1.0,0.0,1.0,0.0,1.0


In [17]:
one_hot= one_hot.reset_index(drop=True)
num_col= num_col.reset_index(drop=True)
# final_df=pd.concat([one_hot,num_col],axis=1)
# final_df.head()

In [18]:
final_df=pd.concat([one_hot,num_col],axis=1)
final_df.head()

,Gender_Male,Customer Type_disloyal Customer,Type of Travel_Personal Travel,Class_Business,Class_Eco,Class_Eco Plus,satisfaction_satisfied,Age,Flight Distance,Inflight entertainment,Baggage handling,Cleanliness,Departure Delay in Minutes,Arrival Delay in Minutes
0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,15,417,1,3,1,8,7.0
1,0.0,0.0,0.0,1.0,0.0,0.0,1.0,23,3832,4,4,4,0,2.0
2,0.0,0.0,1.0,0.0,0.0,1.0,0.0,36,692,4,5,3,163,142.0
3,1.0,0.0,0.0,1.0,0.0,0.0,0.0,45,2163,3,3,2,5,23.0
4,0.0,0.0,1.0,0.0,1.0,0.0,1.0,25,2607,2,3,2,0,0.0


In [19]:
## x and y --> train test split ---> algo traning

In [20]:
# matrix = final_df.corr()
# matrix       

In [21]:
## trainning and testing data
from sklearn.model_selection import train_test_split
## divide the data into x and y or independent and dependent variable
x =final_df.drop("Customer Type_disloyal Customer",axis=1)
y= final_df["Customer Type_disloyal Customer"]
x_train,x_test,y_train,y_test=train_test_split(x,y,
                                                test_size=0.2) 

In [22]:
# Deep Learning

from sklearn.preprocessing import StandardScaler
sc=StandardScaler()
x_train_scaled=sc.fit_transform(x_train)
x_test_scaled=sc.transform(x_test)

In [23]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense      # dense is work of InputLayer outputLayer and hidden layer

In [24]:
x_train.shape[1]

13

In [25]:
## define your ANN model
model = Sequential()
# input layer
## 68 --> no of neurons --> genral connverdation -- 128
model.add(Dense(68, activation='relu', input_dim=(x_train_scaled.shape[1])))  # input layer with 68 neurons and relu activation function  
# hidden layer
model.add(Dense(32, activation='relu'))  # hidden layer with 32 neurons and relu activation function
model.add(Dense(24, activation='relu'))  # hidden layer with 32 neurons and relu activation function
model.add(Dense(12, activation='relu'))  # hidden layer with 32 neurons and relu activation function 
# output layeer
model.add(Dense(1, activation='sigmoid'))  # output layer with 1 neuron and sigmoid activation function for binary classification

## compile the model
model.compile(optimizer='adam',
               loss='binary_crossentropy',
                 metrics=['accuracy'])  # in compie we define the optimizer, loss function and metrics

model.summary()  # to see the summary of the model

c:\Users\asd\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\core\dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 68)             │           952 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,208 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 24)             │           792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 12)             │           300 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 1)              │            13 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,265 (16.66 KB)

 Trainable params: 4,265 (16.66 KB)

 Non-trainable params: 0 (0.00 B)

In [26]:
(13+1)*68 

952

In [27]:
history= model.fit(x_train_scaled, y_train, 
                   epochs=10,
                     validation_data=(x_test_scaled,y_test))  # training the model with 100 epochs and batch size of 32 and validation split of 0.2

Epoch 1/10
973/973 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step - accuracy: 0.8560 - loss: 0.3392 - val_accuracy: 0.9098 - val_loss: 0.2318
Epoch 2/10
973/973 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9059 - loss: 0.2361 - val_accuracy: 0.9143 - val_loss: 0.2171
Epoch 3/10
973/973 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9184 - loss: 0.2157 - val_accuracy: 0.9158 - val_loss: 0.2119
Epoch 4/10
973/973 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9169 - loss: 0.2117 - val_accuracy: 0.9185 - val_loss: 0.2119
Epoch 5/10
973/973 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9189 - loss: 0.2079 - val_accuracy: 0.9222 - val_loss: 0.2025
Epoch 6/10
973/973 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9210 - loss: 0.2079 - val_accuracy: 0.9188 - val_loss: 0.2023
Epoch 7/10
973/973 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9206 - loss: 0.2030 - val_accuracy: 0.9224 - val_loss: 0.1998
Epoch 8/10
973/973 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9244 - loss: 0.1994 - val_accuracy: 0.

In [28]:
## prediction
y_pred = model.predict(x_test_scaled)  # predicting the test data
prediction_label=(y_pred>0.5).astype(int).ravel()  # converting the predicted values to 0 or 1 based on the threshold of 0.5
prediction_label


244/244 ━━━━━━━━━━━━━━━━━━━━ 0s 790us/step


array([0, 0, 1, ..., 1, 0, 0])

In [29]:
model.save('model.h5')  # saving the model



In [30]:
from tensorflow.keras.models import load_model
model_load= load_model('model.h5')  # loading the saved model